# Stage 3: DPO Preference Alignment
**Goal:** Improve response quality using preference data (chosen vs rejected).

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2 — Imports

In [ ]:
import torch
import json
from unsloth import FastLanguageModel, PatchDPOTrainer
from datasets import Dataset
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments

PatchDPOTrainer()   # Unsloth patch for DPO compatibility
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 3 — Configuration

In [ ]:
SFT_ADAPTER_PATH = "./outputs/stage2_sft_adapter"  # Load from Stage 2
MAX_SEQ_LEN      = 512
LOAD_IN_4BIT     = True
LORA_R           = 16
LORA_ALPHA       = 16
LORA_DROPOUT     = 0.05
TARGET_MODS      = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "up_proj", "down_proj", "gate_proj"]
OUTPUT_DIR       = "./outputs/stage3_dpo"
EPOCHS           = 1       # DPO needs fewer epochs than SFT
BATCH_SIZE       = 2
GRAD_ACCUM       = 4
LR               = 5e-5   # Lower LR for DPO alignment
BETA             = 0.1    # DPO temperature — controls deviation from reference model

## Step 4 — Load SFT model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SFT_ADAPTER_PATH,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODS,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)
print("SFT model loaded for DPO training")

## Step 5 — Load preference dataset

In [ ]:
records = []
with open("data/preference_dataset.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#"):
            records.append(json.loads(line))

print(f"Total preference pairs: {len(records)}")
print("Sample:", records[0])

dataset = Dataset.from_dict({
    "prompt"  : [r["prompt"]   for r in records],
    "chosen"  : [r["chosen"]   for r in records],
    "rejected": [r["rejected"] for r in records],
})
print(dataset)

## Step 6 — Train with DPO

In [ ]:
dpo_config = DPOConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    beta                        = BETA,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 5,
    save_strategy               = "epoch",
    warmup_ratio                = 0.1,
    report_to                   = "none",
    max_length                  = MAX_SEQ_LEN,
    max_prompt_length           = 256,
)

trainer = DPOTrainer(
    model     = model,
    args      = dpo_config,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

print("Starting DPO alignment...")
trainer_stats = trainer.train()
print(f"DPO complete. Final loss: {trainer_stats.training_loss:.4f}")

## Step 7 — Save DPO model

In [ ]:
DPO_ADAPTER_PATH = "./outputs/stage3_dpo_adapter"
model.save_pretrained(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)
print(f"DPO adapter saved to: {DPO_ADAPTER_PATH}")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    model.save_pretrained("/content/drive/MyDrive/finance_ft/stage3_dpo_adapter")
    print("DPO adapter also saved to Google Drive")
except:
    print("Not on Colab — skipping Drive save")

## Step 8 — Final inference test (10 questions)

In [ ]:
ALPACA_TEMPLATE_INFERENCE = """Below is a finance question. Write a clear, accurate, and helpful response.

### Question:
{}

### Answer:
"""

FastLanguageModel.for_inference(model)

test_questions = [
    "What is a mutual fund SIP?",
    "What is the difference between a savings account and a fixed deposit?",
    "How does compound interest work?",
    "What is a credit score and why does it matter?",
    "How can I save tax under Section 80C?",
    "What is the repo rate and how does it affect loans?",
    "What is the difference between term insurance and whole life insurance?",
    "How does the stock market work?",
    "What is a demat account and how do I open one?",
    "What is the difference between a mutual fund and an ETF?",
]

dpo_outputs = {}
for q in test_questions:
    prompt = ALPACA_TEMPLATE_INFERENCE.format(q)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.3,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = answer.split("### Answer:")[-1].strip()
    dpo_outputs[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print("-" * 60)